# needtoknow: the load benchmark on every vector store

Runs [`benchmarks/load_bench.py`](https://github.com/muzakkirhussain011/needtoknow/blob/main/benchmarks/load_bench.py) on each store listed below, one after another: one million MS MARCO passages embedded with BAAI/bge-small-en-v1.5, loaded through needtoknow's backend for that store, then recall and latency by permission selectivity and throughput with concurrent users. [`kaggle_stores.sh`](https://github.com/muzakkirhussain011/needtoknow/blob/main/benchmarks/kaggle_stores.sh) starts each store from its release build, since Kaggle has no Docker, and stops it before the next one starts. The pgvector run, with the cost of a remote permission service, is in [`kaggle_load.ipynb`](https://github.com/muzakkirhussain011/needtoknow/blob/main/benchmarks/kaggle_load.ipynb).

Notebook settings: **Internet on**. Add the output of an earlier run that saved the embeddings (such as the pgvector run) as input; without it, choose a **GPU** accelerator for the embeddings. Each store takes about half an hour.

In [ ]:
STORES = ["qdrant", "elasticsearch", "opensearch", "weaviate"]  # the stores to measure, in order
REF = "main"  # the needtoknow branch, tag or commit to benchmark

!rm -rf needtoknow && git clone -q https://github.com/muzakkirhussain011/needtoknow.git
!git -C needtoknow checkout -q {REF} && git -C needtoknow log -1 --format="%H %cs %s"
!pip install -q "./needtoknow[qdrant,weaviate,elasticsearch,opensearch]"

In [ ]:
import shutil
from pathlib import Path

# Embeddings saved by an earlier run (its output attached as input) skip the GPU step.
for saved in Path("/kaggle/input").glob("**/cache/*.npz"):
    Path("/kaggle/working/cache").mkdir(parents=True, exist_ok=True)
    shutil.copy(saved, "/kaggle/working/cache")
    print(f"reusing {saved}")

In [ ]:
import shlex

urls = {
    "qdrant": "http://127.0.0.1:6333",
    "elasticsearch": "http://127.0.0.1:9200",
    "opensearch": "http://127.0.0.1:9200",
    "weaviate": "http://127.0.0.1:8080",
}
for store in STORES:
    !bash needtoknow/benchmarks/kaggle_stores.sh {store}
    args = [
        "--backend",
        store,
        "--url",
        urls[store],
        "--dataset",
        "msmarco",
        "--embedder",
        "BAAI/bge-small-en-v1.5",
        "--query-prefix",
        "Represent this sentence for searching relevant passages: ",
        "--cache",
        "/kaggle/working/cache",
        "--out",
        f"/kaggle/working/load_{store}.json",
        "--markdown",
        f"/kaggle/working/load_{store}.md",
    ]
    command = shlex.join(["python", "needtoknow/benchmarks/load_bench.py", *args])
    !{command}
    !bash needtoknow/benchmarks/kaggle_stores.sh stop

In [ ]:
from IPython.display import Markdown

results = [Path(f"/kaggle/working/load_{store}.md") for store in STORES]
Markdown("\n\n".join(path.read_text() for path in results if path.exists()))